In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("IDS2018-Weighted-v2") \
    .master("spark://lattitude7420:7077") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("UI:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 00:06:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.8
UI: http://lattitude7420:4040


In [2]:
df_2017 = spark.read.csv("hdfs://lattitude7420:9000/ids2017/raw/Monday-WorkingHours.pcap_ISCX.csv",
                          header=True, inferSchema=False)

# Load one 2018 file (already on HDFS)
df_2018 = spark.read.parquet("hdfs://lattitude7420:9000/ids2018/train/")

cols_2017 = set(df_2017.columns)
cols_2018 = set(df_2018.columns)

print("IN 2018 NOT IN 2017:", cols_2018 - cols_2017)
print("IN 2017 NOT IN 2018:", cols_2017 - cols_2018)
print("COMMON:", len(cols_2017 & cols_2018))

IN 2018 NOT IN 2017: {'Subflow Fwd Byts', 'Idle Min', 'Fwd IAT Mean', 'Bwd PSH Flags', 'Active Max', 'PSH Flag Cnt', 'SYN Flag Cnt', 'Fwd Pkt Len Mean', 'Pkt Len Var', 'byte_ratio', 'Fwd Pkt Len Max', 'log_flow_duration', 'Fwd Blk Rate Avg', 'ACK Flag Cnt', 'Fwd Byts/b Avg', 'Bwd Blk Rate Avg', 'Bwd URG Flags', 'Subflow Fwd Pkts', 'Flow IAT Std', 'Pkt Len Std', 'Dst Port', 'Active Min', 'pkt_ratio', 'attack_type', 'Bwd IAT Min', 'Fwd IAT Tot', 'ECE Flag Cnt', 'Flow Duration', 'Fwd Seg Size Avg', 'Protocol', 'Flow IAT Min', 'Active Std', 'Tot Fwd Pkts', 'Bwd Header Len', 'Label', 'day_of_week', 'CWE Flag Count', 'Fwd Pkts/b Avg', 'Pkt Len Min', 'Subflow Bwd Byts', 'Flow IAT Mean', 'binary_label', 'hour_of_day', 'Bwd Pkt Len Std', 'FIN Flag Cnt', 'Bwd IAT Std', 'Bwd IAT Tot', 'Fwd Pkt Len Std', 'Bwd Pkts/b Avg', 'Fwd URG Flags', 'TotLen Bwd Pkts', 'Bwd Seg Size Avg', 'Bwd Pkt Len Max', 'Fwd IAT Std', 'TotLen Fwd Pkts', 'Bwd Pkts/s', 'Bwd Pkt Len Mean', 'Bwd IAT Mean', 'Init Bwd Win Byts'

In [3]:
from pyspark.sql import functions as F

# load 2017
df = spark.read.csv(
    "hdfs://lattitude7420:9000/ids2017/raw/*.csv",
    header=True, inferSchema=True
)

In [4]:
# strip leading spaces from all column names

df = df.toDF(*[c.strip() for c in df.columns])

rename_map = {
    "Total Fwd Packets":       "Tot Fwd Pkts",
    "Total Backward Packets":  "Tot Bwd Pkts",
    "Total Length of Fwd Packets": "TotLen Fwd Pkts",
    "Total Length of Bwd Packets": "TotLen Bwd Pkts",
    "Fwd Packet Length Max":   "Fwd Pkt Len Max",
    "Fwd Packet Length Min":   "Fwd Pkt Len Min",
    "Fwd Packet Length Mean":  "Fwd Pkt Len Mean",
    "Fwd Packet Length Std":   "Fwd Pkt Len Std",
    "Bwd Packet Length Max":   "Bwd Pkt Len Max",
    "Bwd Packet Length Min":   "Bwd Pkt Len Min",
    "Bwd Packet Length Mean":  "Bwd Pkt Len Mean",
    "Bwd Packet Length Std":   "Bwd Pkt Len Std",
    "Fwd IAT Total":           "Fwd IAT Tot",
    "Bwd IAT Total":           "Bwd IAT Tot",
    "Fwd Packets/s":           "Fwd Pkts/s",
    "Bwd Packets/s":           "Bwd Pkts/s",
    "Min Packet Length":       "Pkt Len Min",
    "Max Packet Length":       "Pkt Len Max",
    "Packet Length Mean":      "Pkt Len Mean",
    "Packet Length Std":       "Pkt Len Std",
    "Packet Length Variance":  "Pkt Len Var",
    "ACK Flag Count":          "ACK Flag Cnt",
    "URG Flag Count":          "URG Flag Cnt",
    "FIN Flag Count":          "FIN Flag Cnt",
    "PSH Flag Count":          "PSH Flag Cnt",
    "RST Flag Count":          "RST Flag Cnt",
    "SYN Flag Count":          "SYN Flag Cnt",
    "ECE Flag Count":          "ECE Flag Cnt",
    "Average Packet Size":     "Pkt Size Avg",
    "Avg Fwd Segment Size":    "Fwd Seg Size Avg",
    "Avg Bwd Segment Size":    "Bwd Seg Size Avg",
    "Bwd Header Length":       "Bwd Header Len",
    "Subflow Fwd Bytes":       "Subflow Fwd Byts",
    "Subflow Fwd Packets":     "Subflow Fwd Pkts",
    "Subflow Bwd Bytes":       "Subflow Bwd Byts",
    "Subflow Bwd Packets":     "Subflow Bwd Pkts",
    "Init_Win_bytes_forward":  "Init Fwd Win Byts",
    "Init_Win_bytes_backward": "Init Bwd Win Byts",
    "act_data_pkt_fwd":        "Fwd Act Data Pkts",
    "min_seg_size_forward":    "Fwd Seg Size Min",
    "Destination Port":        "Dst Port",
    "Fwd Avg Bulk Rate":       "Fwd Blk Rate Avg",
    "Bwd Avg Bulk Rate":       "Bwd Blk Rate Avg",
    "Fwd Avg Bytes/Bulk":      "Fwd Byts/b Avg",
    "Bwd Avg Bytes/Bulk":      "Bwd Byts/b Avg",
    "Fwd Avg Packets/Bulk":    "Fwd Pkts/b Avg",
    "Bwd Avg Packets/Bulk":    "Bwd Pkts/b Avg",
}

for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

In [5]:
if "Fwd Header Length34" in df.columns:
    df = df.withColumn("Fwd Header Len",
            F.coalesce(F.col("Fwd Header Length34"), F.col("Fwd Header Length55"))
          ).drop("Fwd Header Length34", "Fwd Header Length55")

# fill missing cols
df = df.withColumn("Protocol",      F.lit(0))
df = df.withColumn("Fwd PSH Flags", F.lit(0))

In [6]:
# engineered features
df = df.withColumn("log_flow_duration", F.log1p(F.col("Flow Duration")))

df = df.withColumn("pkt_ratio",
        F.when((F.col("Tot Fwd Pkts") + F.col("Tot Bwd Pkts")) > 0,
               F.col("Tot Fwd Pkts") / (F.col("Tot Fwd Pkts") + F.col("Tot Bwd Pkts"))
        ).otherwise(0.0))

df = df.withColumn("byte_ratio",
        F.when((F.col("TotLen Fwd Pkts") + F.col("TotLen Bwd Pkts")) > 0,
               F.col("TotLen Fwd Pkts") / (F.col("TotLen Fwd Pkts") + F.col("TotLen Bwd Pkts"))
        ).otherwise(0.0))

In [7]:
if "Timestamp" in df.columns:
    df = df.withColumn("ts_parsed", F.to_timestamp("Timestamp", "dd/MM/yyyy HH:mm"))
    df = df.withColumn("hour_of_day",  F.hour("ts_parsed"))
    df = df.withColumn("day_of_week",  F.dayofweek("ts_parsed"))
    df = df.drop("ts_parsed", "Timestamp")

In [8]:
[f for f in cols_2018 if f not in df.columns]

['attack_type', 'day_of_week', 'binary_label', 'hour_of_day']

In [9]:
[f for f in df.columns if f not in cols_2018]

['Flow Bytes/s', 'Flow Packets/s']

In [10]:
# fill missing time features — no Timestamp in 2017
df = df.withColumn("hour_of_day", F.lit(0))
df = df.withColumn("day_of_week", F.lit(0))

# keep Label for GUI ground truth, drop only junk cols
drop_cols = ["Flow Bytes/s", "Flow Packets/s"]
df_clean = df.drop(*[c for c in drop_cols if c in df.columns])

In [14]:
[f for f in cols_2018 if f not in df_clean.columns]

['attack_type', 'binary_label']

In [15]:
[f for f in df_clean.columns if f not in cols_2018]

[]

In [17]:
final_features = cols_2018
missing = [f for f in final_features if f not in df_clean.columns]
print("MISSING FROM MODEL FEATURES:", missing)  # must be empty
print("Total rows:", df_clean.count())

MISSING FROM MODEL FEATURES: ['attack_type', 'binary_label']


[Stage 4:===================================================>     (10 + 1) / 11]

Total rows: 2830743


In [26]:
df_clean = df_clean.withColumn("Label",
    F.when(F.col("Label") == "BENIGN", "Benign")
     .when(F.col("Label").startswith("Web Attack"), "WebAttack")
     .when(F.col("Label") == "SSH-Patator", "SSH-Bruteforce")
     .when(F.col("Label") == "FTP-Patator", "FTP-BruteForce")
     .when(F.col("Label").startswith("DoS"), "DoS")
     .when(F.col("Label") == "DDoS", "DDoS")
     .when(F.col("Label") == "Bot", "Bot")
     .when(F.col("Label") == "Infiltration", "Infilteration")
     .when(F.col("Label") == "Heartbleed", "DoS")
     .when(F.col("Label") == "PortScan", "Unknown")
     .otherwise("Unknown")
)

In [28]:
df_clean.groupBy("Label").count().orderBy("count", ascending=False).show()

[Stage 14:==============================================>          (9 + 2) / 11]

+--------------+-------+
|         Label|  count|
+--------------+-------+
|        Benign|2273097|
|           DoS| 252672|
|       Unknown| 158930|
|          DDoS| 128027|
|FTP-BruteForce|   7938|
|SSH-Bruteforce|   5897|
|     WebAttack|   2180|
|           Bot|   1966|
| Infilteration|     36|
+--------------+-------+



In [29]:
df_clean.write.mode("overwrite").parquet("hdfs://lattitude7420:9000/ids2017/cleaned")
print("Saved.")

[Stage 17:==================================================>     (10 + 1) / 11]

Saved.


In [30]:
spark.stop()